# Normalize attention one tile at a time

Machine: CPU, ordinary PyTorch autograd. Time: 45–60 minutes. Prerequisites: Chapter4 softmax and causal attention; resource accounting companion. Deliverable: a traced online update, independent output/gradient comparison, and a broken-variant diagnosis.

A full row of scores is convenient but expensive to materialize. Can a row arrive in pieces without changing its normalized weighted value? Track a running maximum m, denominator l and value numerator u. All statistics must use the same exponential units. This original Python reference teaches the recurrence; it is not a fused GPU kernel or a backward-memory optimization. Query/key/value shapes are[batch,heads,length,width].

## 1. A late maximum changes the units of old work

### Prediction

For scores[0,1000] and values[0,1], predict the output. If the running maximum jumps to1000, what should happen to the earlier denominator and numerator?

### Checkpoint

Compare stable rescaling, a deliberately omitted rescale, and direct exponentiation. Trace m,l,u after each key tile.

### Reference solution

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.nn import functional as F
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'src/dongxi_llms').is_dir());sys.path.insert(0,str(ROOT/'src'))
from dongxi_llms.online_attention_lab import blockwise_attention,dense_attention,unstable_attention,forward_workspace
from dongxi_llms.systems_performance_lab import saved_forward_storage
q=torch.ones(1,1,1,1,dtype=torch.float64);k=torch.tensor([0.,1000.],dtype=torch.float64).reshape(1,1,2,1);v=torch.tensor([0.,1.],dtype=torch.float64).reshape(1,1,2,1)
out,lse,trace=blockwise_attention(q,k,v,causal=False,key_tile=1,trace=True)
broken,_,_=blockwise_attention(q,k,v,causal=False,key_tile=1,rescale_old=False)
print('stable:',out.item(),'omitted rescale:',broken.item(),'direct exp finite:',bool(torch.isfinite(unstable_attention(q,k,v)).all()))
fig,axes=plt.subplots(1,3,figsize=(11,3))
for ax,key,label in zip(axes,['row_max','normalizer','value_numerator'],['Running maximum m','Normalizer l','Weighted numerator u']):
    vals=[np.asarray(t[key]).reshape(-1)[0] for t in trace];ax.plot([1,2],vals,'o-');ax.set(xlabel='Processed key tile',ylabel=label);ax.set_xticks([1,2])
plt.tight_layout();plt.show();print(trace)

### Reference explanation

The correct output is numerically1. When m changes from0 to1000, old l and u are multiplied by exp(0−1000), numerically0 in FP64; new weights use exp(score−1000). The broken control leaves old l=1 then adds another1, producing output1/2. Direct exp(1000) overflows. In general, m′=max(m,max tile scores), l′=exp(m−m′)l+Σexp(s−m′), and u′=exp(m−m′)u+Σexp(s−m′)v. Final output=u/l; logsumexp=m+log l. The plot is a traced authored fixture, not measured model attention.

![Reference plot: 1. A late maximum changes the units of old work. Axis labels identify the mechanism quantities.](../../figures/chapter-04/cs336-02_online_attention-01.png)

Read this preview alongside the explanation above. Regenerate it by running the adjacent reference cell; simulated curves and retained measurements keep their stated evidence labels.


## 2. Tile flow must respect causal visibility

### Prediction

If key/value positions3 onward change, which earlier queries can change under the mask k≤q? Predict the answer before looking at the matrix.

### Checkpoint

Compare online output with independently called PyTorch SDPA and compare gradients of sum(output²). Then perturb only future values.

### Reference solution

In [ ]:
g=torch.Generator().manual_seed(6505)
q,k,v=[torch.randn(1,1,5,3,generator=g,dtype=torch.float64,requires_grad=True) for _ in range(3)]
mask=torch.arange(5)[None,:]<=torch.arange(5)[:,None]
reference=F.scaled_dot_product_attention(q,k,v,attn_mask=mask,dropout_p=0.)
result,lse,flow=blockwise_attention(q,k,v,query_tile=2,key_tile=2,trace=True)
grad_ref=torch.autograd.grad(reference.square().sum(),(q,k,v),retain_graph=True)
grad_online=torch.autograd.grad(result.square().sum(),(q,k,v),retain_graph=True)
print('output max error:',(reference-result).detach().abs().max().item(),'gradient max error:',max((a-b).abs().max().item() for a,b in zip(grad_ref,grad_online)))
changed=v.detach().clone();changed[:,:,3:]+=100
perturbed,_,_=blockwise_attention(q,k,changed,query_tile=2,key_tile=2)
print('first3 query output change:',(perturbed[:,:,:3]-result[:,:,:3]).detach().abs().max().item())
fig,ax=plt.subplots(figsize=(5,4));im=ax.imshow(mask.numpy(),origin='upper',vmin=0,vmax=1,cmap='Blues');
for boundary in [1.5,3.5]:ax.axvline(boundary,color='orange');ax.axhline(boundary,color='orange')
ax.set(xlabel='Key position',ylabel='Query position',title='Allowed causal entries; orange tile boundaries');fig.colorbar(im,ax=ax,label='Visible (1) / masked (0)');plt.show()

### Reference explanation

Queries0–2 cannot read keys3–4, so their output stays unchanged. Orange boundaries slice computation without changing the blue causal region; the final partial tile still obeys absolute positions. FP64 output/gradient tolerance is1e-11; FP32 is3e-6 in the independent tests. Ordinary autograd follows all running-statistic operations. This parity proves the differentiable mechanism on tested shapes, not GPU speed or reduced backward memory. Change query/key tiles to1 or oversized values to test that storage partitions do not become semantic partitions.

![Reference plot: 2. Tile flow must respect causal visibility. Axis labels identify the mechanism quantities.](../../figures/chapter-04/cs336-02_online_attention-02.png)

Read this preview alongside the explanation above. Regenerate it by running the adjacent reference cell; simulated curves and retained measurements keep their stated evidence labels.


## 3. Padding and an empty row have an explicit contract

### Prediction

If a sequence has no valid keys, should output be NaN, zero, or uniform over padding? What is its row logsumexp?

### Checkpoint

Use two batch members with different lengths. Compare a cached query offset with independent SDPA. Display invalid rows and verify their zero contribution.

### Reference solution

In [ ]:
g=torch.Generator().manual_seed(6506)
q=torch.randn(2,2,3,4,generator=g,dtype=torch.float64,requires_grad=True);k=torch.randn(2,2,7,4,generator=g,dtype=torch.float64,requires_grad=True);v=torch.randn(2,2,7,2,generator=g,dtype=torch.float64,requires_grad=True)
ql=torch.tensor([2,3]);kl=torch.tensor([6,0]);qi=torch.arange(3)[None,:,None];ki=torch.arange(7)[None,None,:]
allowed=(qi<ql[:,None,None])&(ki<kl[:,None,None])&(ki<=qi+4)
reference=F.scaled_dot_product_attention(q,k,v,attn_mask=allowed[:,None])
result,lse,_=blockwise_attention(q,k,v,query_lengths=ql,key_lengths=kl,query_offset=4,query_tile=2,key_tile=3)
print('independent cached/ragged max error:',(reference-result).detach().abs().max().item());print('LSE including invalid rows:',lse.detach())
fig,axes=plt.subplots(1,2,figsize=(9,3))
for b,ax in enumerate(axes):
    im=ax.imshow(result[b,0].detach(),aspect='auto',cmap='coolwarm');ax.set(xlabel='Value coordinate',ylabel='Stored query position',title=f'Batch{b}: valid Q={ql[b].item()},K={kl[b].item()}');fig.colorbar(im,ax=ax,label='Output')
plt.tight_layout();plt.show()
gradients=torch.autograd.grad(result.square().sum(),(q,k,v));print('All finite gradients:',all(bool(torch.isfinite(t).all()) for t in gradients),'empty member gradient sum:',sum(t[1].abs().sum().item() for t in gradients))

### Reference explanation

The contract is output0 and logsumexp−∞ for a fully masked or invalid query. There is no probability mass over padding. Query offset4 means stored query0 represents absolute position4; it may see keys0–4. Safe branches keep empty accumulators from dividing0/0, and masked coordinates contribute zero gradient to this objective. The colored matrix displays stored rows, including invalid padding; those zeros are an intentional result, not evidence of a learned behavior.

![Reference plot: 3. Padding and an empty row have an explicit contract. Axis labels identify the mechanism quantities.](../../figures/chapter-04/cs336-02_online_attention-03.png)

Read this preview alongside the explanation above. Regenerate it by running the adjacent reference cell; simulated curves and retained measurements keep their stated evidence labels.


## 4. A smaller forward tile is not a backward-memory result

### Prediction

With fixed tiles, which logical intermediate stays bounded as sequence length grows? Can ordinary autograd still retain many intermediate tensors?

### Checkpoint

Plot forward-workspace estimates separately from instrumented forward saves. Do not label either as process peak or FlashAttention training memory.

### Reference solution

In [ ]:
lengths=[8,16,32,64];workspace=[forward_workspace(1,1,x,x,4,query_tile=4,key_tile=4) for x in lengths]
saves=[]
for length in lengths:
    g=torch.Generator().manual_seed(6507);q,k,v=[torch.randn(1,1,length,4,generator=g,dtype=torch.float64,requires_grad=True) for _ in range(3)]
    dense_result,dense_saved=saved_forward_storage(lambda:dense_attention(q,k,v)[0])
    online_result,online_saved=saved_forward_storage(lambda:blockwise_attention(q,k,v,query_tile=4,key_tile=4)[0])
    saves.append((dense_saved['unique_backing_storage_bytes'],online_saved['unique_backing_storage_bytes']))
fig,axes=plt.subplots(1,2,figsize=(11,3));axes[0].plot(lengths,[r['dense_score_probability_bytes']/1024 for r in workspace],'o-',label='dense score/probability arrays');axes[0].plot(lengths,[(r['block_score_weight_bytes']+r['block_accumulator_statistics_bytes'])/1024 for r in workspace],'o-',label='block score/weights + accumulators');axes[0].set(xlabel='Sequence length',ylabel='Modeled workspace (KiB)',title='Logical forward intermediates; fixed4×4 tiles');axes[0].legend()
for i,label in enumerate(['dense ordinary autograd','online ordinary autograd']):axes[1].plot(lengths,[r[i]/1024 for r in saves],'o-',label=label)
axes[1].set(xlabel='Sequence length',ylabel='Saved backing storage (KiB)',title='Actual forward saves; not peak/liveness');axes[1].legend();plt.tight_layout();plt.show()
print('Forward-save storage bytes:',list(zip(lengths,saves)))

### Reference explanation

The score/weight tile and per-row accumulator are bounded with fixed tiles and width, whereas dense scores and probabilities grow quadratically. Input and output storage still grow with sequence length. The second panel records ordinary autograd saves: loops may retain many small tensors, so this implementation can save more storage than dense attention. A recomputed custom backward would need its own implementation, gradient tests and memory measurements; it is outside this reference. Do not infer training-memory savings from the left panel.

![Reference plot: 4. A smaller forward tile is not a backward-memory result. Axis labels identify the mechanism quantities.](../../figures/chapter-04/cs336-02_online_attention-04.png)

Read this preview alongside the explanation above. Regenerate it by running the adjacent reference cell; simulated curves and retained measurements keep their stated evidence labels.


## 5. Choose the failed explanation

### Prediction

A colleague says: “Our outputs match SDPA, therefore we reproduced FlashAttention speed.” Which missing evidence would change the conclusion?

### Checkpoint

Try oversized tiles and a noncausal control. Separate mask semantics, numerical stability, output/gradient parity, modeled workspace and actual execution.

### Reference solution

In [ ]:
g=torch.Generator().manual_seed(6508);q,k,v=[torch.randn(1,1,4,2,generator=g,dtype=torch.float64) for _ in range(3)]
causal,_,_=blockwise_attention(q,k,v,query_tile=9,key_tile=9)
noncausal,_,_=blockwise_attention(q,k,v,query_tile=1,key_tile=3,causal=False)
expected=F.scaled_dot_product_attention(q,k,v,is_causal=False)
fig,ax=plt.subplots(figsize=(6,3));ax.plot(range(4),(causal-noncausal).abs().sum(-1)[0,0].numpy(),'o-');ax.set(xlabel='Query position',ylabel='L1 output difference',title='Removing a mask changes semantics, not just speed');plt.show()
print('Noncausal independent max error:',(expected-noncausal).abs().max().item())

### Reference explanation

Mask changes alter available evidence and usually outputs; tile changes under the same mask should not. Matching a reference establishes correctness within declared tolerances and tested boundaries. A speed claim additionally needs an actual device, kernel, shapes, dtype, warmup, synchronization, repetitions and timed phase. This CPU loop makes no such GPU-performance claim. Neither a backend name nor fewer modeled intermediates establishes a fused implementation.

![Reference plot: 5. Choose the failed explanation. Axis labels identify the mechanism quantities.](../../figures/chapter-04/cs336-02_online_attention-05.png)

Read this preview alongside the explanation above. Regenerate it by running the adjacent reference cell; simulated curves and retained measurements keep their stated evidence labels.


## Evidence and next step

Defend the recurrence by showing why old and new contributions share units after each update. Submit a masked-row test and the broken-rescale diagnosis. The [focused solutions](SYSTEMS_SOLUTIONS.md) map this to AssignmentC. The next notebook asks a different question: what can a compute budget predict when the observations cover only a small grid?